<div style="text-align: center;">
<a target="_blank" href="https://colab.research.google.com/github/miquelmn/aa_2526/blob/main/06_Transfer/AlexNet_Transfer.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
</div>

# Models ja existents i *transfer learning*

En aquesta pràctica, aprofundirem en la classificació d’imatges amb xarxes neuronals convolucionals (CNNs), però amb un enfocament diferent al de la sessió anterior. Mentre que anteriorment vàrem construir CNNs des de zero per comprendre la seva estructura bàsica, aquest cop treballarem amb un model de CNN preentrenat: **AlexNet**. Els objectius són:

- **Comprendre i utilitzar un model existent**: en aquest cas, AlexNet, un model ja entrenat sobre un gran conjunt de dades.
- **Transfer Learning**: aprendre com aprofitar els coneixements d’una xarxa preentrenada i adaptar-la per resoldre una nova tasca.
- **Càrrega de conjunts de dades d’imatges locals**: aplicar el processament d’imatges i la càrrega de dades des de l’ordinador.

Aquest enfocament ajuda a optimitzar l’entrenament i és especialment útil quan es disposa de pocs recursos computacionals o un conjunt de dades més reduït.

### AlexNet

AlexNet és una xarxa que va establir un nou estàndard en visió per computador gràcies a la seva capacitat de reconeixement en múltiples categories. La seva estructura, composta de capes convolucionals i de max-pooling, amb capes totalment connectades al final, és una referència en el camp de les CNNs.

Aquest cop, usarem AlexNet com a punt de partida, sense construir el model des de zero, per veure com es poden reutilitzar i adaptar les seves característiques apreses per a nous conjunts de dades.

### Què és el Transfer Learning?

El Transfer Learning és una tècnica que permet aprofitar les xarxes preentrenades (com AlexNet) per a una nova tasca. La xarxa es modifica per adaptar-la a les noves classes del conjunt de dades que volem classificar, fent ús de les característiques generals ja apreses en l’entrenament inicial (vores, textures, etc.).

Aquest process es pot fer de dues maneres. La primera és el que també rep el nom de **fine-tunning**:
- Congelarem les primeres capes del model per conservar les característiques generals apreses.
- Modificarem i entrenarem només les capes finals per adaptar-les a les noves classes, fent que el model s’ajusti de forma ràpida i amb menys dades.

La segona, que anomenam també com la categoria general **transfer learning**:

- Congelarem les capes de l'extractor de característiques del model per conservar les característiques generals apreses.
- Afegir un nou classificador ``MLP`` i entrenar-ho de 0.


Aquest procediment permetrà entendre com es pot utilitzar una xarxa ja existent per resoldre tasques específiques sense haver de construir ni entrenar un model completament des de zero.

### Guia de la Pràctica

En aquest notebook treballarem per:

 1. Carregar i preparar un conjunt d’imatges des de l’ordinador.
 2. Utilitzar el model AlexNet preentrenat i aplicar transfer learning per ajustar-lo a noves categories.
 3. Analitzar el rendiment del model i visualitzar els resultats.

## Començam

Primer de tot, com sempre, hem d'obtenir les dades. Aquesta sessió la farem amb el conjunt de dades [Tiny ImageNet](https://www.kaggle.com/c/tiny-imagenet/data?select=train.images.zip).

Aquest conjunt de dades es defineix en la seva plana de la forma següent:

> MicroImageNet classification challenge is similar to the classification challenge in the full ImageNet ILSVRC. MicroImageNet contains 200 classes for training. Each class has 500 images. The test set contains 10,000 images. All images are 64x64 colored ones.

Aquesta vegada no farem feina amb un conjunt de dades ja existents a ``torchvision`` sinó que nosaltres farem la gestió des de 0. Per tant i primer de tot descarregarem les dades. Per fer-ho podem trobar el conjunt de dades a la següent plana (http://cs231n.stanford.edu/tiny-imagenet-200.zip).

Alternativament també podem emprar l'eina ``wget`` així:

```
wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
```

Una vegada que hem descarregat les dades les podem descomprimir i finalment comença a fer-hi feina.

## Definició de la xarxa: AlexNet i *Transfer learning*

En aquesta pràctica aplicarem la tècnica de transfer learning amb la primera xarxa CNN moderna:
- AlexNet. [ImageNet Classification with Deep Convolutional Neural Network, 2012](https://proceedings.neurips.cc/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf). La mida d'entrada de les imatges és de (227x227x3).Té prop de 60 milions de paràmetres entrenables.

Pytorch ens permet emprar aquest tipus de xarxes de manera molt senzilla. [Més informació](https://pytorch.org/vision/stable/models.html). Si el model que cercam no es troba integrat dins la llibreria Pytorch és bastant probable que si la trobem a Huggingface.

Descarregarem AlexNet i a analitzar-la. En aquest cas no només ens baixam la seva arquitectura, també els pesos resultants de l'entrenament.

**Normalment els problems els resoldrem emprant models ja definits i preentrenats**




## Feina
----------------------------------------------------------------------
**1.Carregar la xarxa AlexNet i congelar l'extractor de característiques**

In [1]:
from tqdm.auto import tqdm


import torch
from torch import nn
import torch.optim as optim
from torchvision import datasets, models, transforms

Descarregam el conjunt de dades per poder emplearlo per entrenar la xarxa

In [2]:
!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip

"wget" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


Extreim el conjunt de dades del zip

In [3]:
!unzip tiny-imagenet-200.zip

"unzip" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


**Batch Normalization**

In [4]:
class MyBatchNorm1d(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        self.num_features = num_features

        # Evitar divisió per 0
        self.eps = eps

        # Momentum
        self.momentum = momentum

        # Paràmetres aprenables: gamma (weight) i beta (bias)
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn.Parameter(torch.zeros(num_features))

        # Estadístiques que s’acumulen durant l’entrenament però que no són paràmetres
        self.register_buffer("running_mean", torch.zeros(num_features))
        self.register_buffer("running_var", torch.ones(num_features))

    def forward(self, x):
        if self.training:
            # Calcular mitjana i variància del batch
            batch_mean = x.mean(dim=0)
            batch_var = x.var(dim=0, unbiased=False)

            # Actualitzar estadístiques globals
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * batch_mean
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * batch_var

            # Normalitzar el batch actual
            x_hat = (x - batch_mean) / torch.sqrt(batch_var + self.eps)
        else:
            # En mode d’avaluació, s’usen les estadístiques acumulades
            x_hat = (x - self.running_mean) / torch.sqrt(self.running_var + self.eps)

        # Aplicar gamma i beta
        y = self.gamma * x_hat + self.beta
        return y

Cream el transform per les dades i dividim el conjunt de dades entre train i test

In [5]:
BATCH_SIZE = 16
EPOCHS = 10

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train = datasets.ImageFolder('tiny-imagenet-200/train', transform=transform)
test = datasets.ImageFolder('tiny-imagenet-200/val', transform=transform) # Note: using 'val' as per the dataset structure

train_loader = torch.utils.data.DataLoader(train,
                                           batch_size=BATCH_SIZE,
                                           shuffle=True)
test_loader = torch.utils.data.DataLoader(test,
                                          batch_size=BATCH_SIZE,
                                          shuffle=True)

FileNotFoundError: [WinError 3] El sistema no puede encontrar la ruta especificada: 'tiny-imagenet-200/train'

In [ ]:
img, target = next(iter(train_loader))
print(img.shape, target)

torch.Size([16, 3, 224, 224]) tensor([195, 136, 193,  13,  85,  66, 182, 143,  93, 100, 141,   1,   8,  15,
        130,  70])


Carregam la Alex Net amb els pesos, perque no els haguem de calcular un altre cop

In [ ]:
alex = models.alexnet(weights=True)

print("-" * 50)
print("Arquitectura AlexNet")
print("-" * 50)
print(alex)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 186MB/s]


--------------------------------------------------
Arquitectura AlexNet
--------------------------------------------------
AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classi

In [ ]:
for param in alex.features.parameters():
   param.requires_grad = False

**2.Definir un entorn seqüencial on implementarem el classificador de la xarxa.**

Hem afegit despres de cada capa *Lineal* la capa de *MyBatchNorm1d*

In [ ]:
in_features = alex.classifier[1].in_features
num_classes = len(train.classes) # Get the number of classes from the training dataset
print(f"Nombre de d'entrades: ", in_features)
print(f"Nombre de classes de sortida: ", num_classes)

# Define a new classifier with Batch Normalization layers using MyBatchNorm1d
classifier = nn.Sequential(
    nn.Dropout(p=0.5, inplace=False),
    nn.Linear(in_features, 4096),
    nn.BatchNorm1d(4096), # Add MyBatchNorm1d after the first linear layer
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.5, inplace=False),
    nn.Linear(4096, 4096),
    nn.BatchNorm1d(4096), # Add MyBatchNorm1d after the second linear layer
    nn.ReLU(inplace=True),
    nn.Linear(4096, num_classes) # Output features equal to the number of classes
)

# Replace the original classifier with the new one
alex.classifier = classifier

print("-" * 50)
print("AlexNet con nuevo classificador (con MyBatchNorm1d)")
print("-" * 50)
print(alex)

Nombre de d'entrades:  9216
Nombre de classes de sortida:  200
--------------------------------------------------
AlexNet con nuevo classificador (con MyBatchNorm1d)
--------------------------------------------------
AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0,

**3.Realitzar un entrenament: comparar rendiment (accuracy) i nombre de paràmetres.**

In [ ]:
# Check for GPU availability and set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
alex.to(device)
print(f"Using device: {device}")

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(alex.classifier.parameters(), lr=0.001, momentum=0.9) # Only optimize the new classifier

# Training loop
for epoch in range(EPOCHS):
    running_loss = 0.0
    correct = 0
    total = 0
    for i, data in tqdm(enumerate(train_loader, 0), total=len(train_loader)):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device) # Move data to GPU

        optimizer.zero_grad()

        outputs = alex(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Calculate accuracy for the mini-batch
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # Print loss and accuracy every 1000 mini-batches
        if i % 1000 == 999:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 1000:.3f}, accuracy: {100 * correct / total:.2f} %')
            # Reset running_loss, correct, and total for the next interval
            running_loss = 0.0
            correct = 0
            total = 0

print('Finished Training')

Using device: cuda


  0%|          | 0/6250 [00:00<?, ?it/s]

[1,  1000] loss: 4.361, accuracy: 11.04 %
[1,  2000] loss: 3.219, accuracy: 26.25 %
[1,  3000] loss: 2.892, accuracy: 31.96 %
[1,  4000] loss: 2.774, accuracy: 34.52 %
[1,  5000] loss: 2.657, accuracy: 36.41 %
[1,  6000] loss: 2.609, accuracy: 37.42 %


  0%|          | 0/6250 [00:00<?, ?it/s]

[2,  1000] loss: 2.315, accuracy: 43.73 %
[2,  2000] loss: 2.342, accuracy: 42.78 %
[2,  3000] loss: 2.334, accuracy: 43.39 %
[2,  4000] loss: 2.357, accuracy: 43.22 %
[2,  5000] loss: 2.321, accuracy: 43.80 %
[2,  6000] loss: 2.320, accuracy: 43.71 %


...

[8,  1000] loss: 1.409, accuracy: 62.17 %
[8,  2000] loss: 1.444, accuracy: 61.61 %
[8,  3000] loss: 1.492, accuracy: 60.27 %
[8,  4000] loss: 1.511, accuracy: 59.65 %
[8,  5000] loss: 1.564, accuracy: 59.28 %
[8,  6000] loss: 1.546, accuracy: 59.14 %


  0%|          | 0/6250 [00:00<?, ?it/s]

[9,  1000] loss: 1.325, accuracy: 64.31 %
[9,  2000] loss: 1.366, accuracy: 62.85 %
[9,  3000] loss: 1.416, accuracy: 61.84 %
[9,  4000] loss: 1.433, accuracy: 61.34 %
[9,  5000] loss: 1.428, accuracy: 62.12 %
[9,  6000] loss: 1.469, accuracy: 60.84 %


  0%|          | 0/6250 [00:00<?, ?it/s]

[10,  1000] loss: 1.250, accuracy: 65.32 %
[10,  2000] loss: 1.285, accuracy: 65.01 %
[10,  3000] loss: 1.329, accuracy: 63.83 %
[10,  4000] loss: 1.332, accuracy: 64.14 %
[10,  5000] loss: 1.377, accuracy: 62.95 %
[10,  6000] loss: 1.379, accuracy: 62.91 %
Finished Training


In [ ]:
# Evaluate the trained model on the test set
correct = 0
total = 0
with torch.no_grad():
    for data in tqdm(test_loader, total=len(test_loader)):
        images, labels = data
        images, labels = images.to(device), labels.to(device) # Move data to GPU
        outputs = alex(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy de la red Modificada: {100 * correct / total:.2f} %')

# Calculate and print the number of parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Nombre de paràmetres de l'AlexNet Modificada: {count_parameters(alex.classifier)}")
original_alex = models.alexnet(weights=True)
print(f"Nombre de paràmetres de l'AlexNet Original: {sum(p.numel() for p in original_alex.parameters())}")

  0%|          | 0/625 [00:00<?, ?it/s]

Accuracy de la red Modificada: 0.70 %
Nombre de paràmetres de l'AlexNet Modificada: 55369928
Nombre de paràmetres de l'AlexNet Original: 61100840


**4.Provar de guardar la vostra xarxa i tornar-la a carregar. Classificar una imatge del conjunt de test.**

In [ ]:
# Guardam la nostra xarxa d'AlexNet
torch.save(alex, 'alexnet_transfer_learning_complete.pth')
print("Modelo completo guardado correctamente!")

# CARGARLO
loaded_alexnet = torch.load(
    'alexnet_transfer_learning_complete.pth',
    map_location=device,
    weights_only=False  # Permite cargar el modelo completo
)
loaded_alexnet.eval()
print("Modelo cargado correctamente!")

Modelo completo guardado correctamente!
Modelo cargado correctamente!


In [ ]:
# Recreate the classifier structure to match the saved model
in_features = loaded_alexnet.classifier[1].in_features
classifier = nn.Sequential(
    nn.Dropout(p=0.5, inplace=False),
    nn.Linear(in_features, 4096),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.5, inplace=False),
    nn.Linear(4096, 4096),
    nn.ReLU(inplace=True),
    nn.Linear(4096, 200)
)
loaded_alexnet.classifier = classifier

# Load the saved state dictionary
loaded_alexnet.load_state_dict(torch.load('alexnet_transfer_learning.pth'))
loaded_alexnet.eval() # Set the model to evaluation mode

# Move the loaded model to the same device as training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_alexnet.to(device)

print("Model Carregat Correctament!")

NameError: name 'loaded_alexnet' is not defined

In [ ]:
import random
import torch

# Inspect the classes to understand the issue
print("Classes in train dataset:", train.classes)
print("Number of classes in train dataset:", len(train.classes))
print("Classes in test dataset:", test.classes)
print("Number of classes in test dataset:", len(test.classes))


# Escogemos una imagen aleatoria del conjunto de test
random_index = random.randint(0, len(test) - 1)
image, label = test[random_index]
print(f"-- Imagen de test seleccionada con label: {label}")

# Añadimos la dimensión de batch y movemos al dispositivo
image = image.unsqueeze(0).to(device)

# Inferencia
with torch.no_grad():
    output = loaded_alexnet(image)
    _, predicted_class = torch.max(output.data, 1)

pred_index = int(predicted_class.item())
true_index = int(label)

# Comprobamos que el índice esté dentro del rango y usamos train.classes
if 0 <= pred_index < len(train.classes):
    predicted_label = train.classes[pred_index]
else:
    predicted_label = f"Índice fuera de rango ({pred_index})"

# Use train.classes for the true label as well, assuming consistent mapping
if 0 <= true_index < len(train.classes):
    true_label = train.classes[true_index]
else:
     true_label = f"Índice fuera de rango ({true_index})"


print(f"\nTrue label: {true_label}")
print(f"Predicted label: {predicted_label}")